# Phase 2: Forecasting Diagnostics

## The Hook
While machine learning models often beat simple benchmarks, our forecasting showdown revealed a surprising result: **The Naive Baseline outperformed LightGBM and Prophet on the 12-week holdout.**

This notebook performs a forensic diagnosis of *why* this occurred by diving deep into time series decomposition, autocorrelation, and feature variance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- HIGH PRIORITY PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data
from src.data.aggregator import aggregate_to_weekly_chain

transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
df_weekly = aggregate_to_weekly_chain(df)
df_weekly.tail()

## 1. Time Series Autocorrelation (ACF & PACF)
Before building features like `Lag_1` or `Lag_52`, a data scientist must prove there's statistical memory. ACF shows correlation with past lags, and PACF shows the direct effect of a lag (controlling for intermediate lags).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

plot_acf(df_weekly['Sales'], lags=52, ax=axes[0], title='Autocorrelation (ACF) - Memory & Seasonality')
plot_pacf(df_weekly['Sales'], lags=52, ax=axes[1], title='Partial Autocorrelation (PACF) - Direct Lag Impact')

plt.tight_layout()
plt.show()

## 2. Seasonal Decomposition
Let's break the chain-level trend down into overarching trajectory, repeating patterns (the 52-week cycle), and noise/residuals. This is exactly what additive models like Prophet attempt to learn.

In [ ]:
# Using a 52-week period for yearly seasonality
decomp = seasonal_decompose(df_weekly['Sales'].values, model='additive', period=52)

fig = decomp.plot()
fig.set_size_inches(14, 10)
plt.suptitle('Chain-Level Sales Decomposition (52-Week Period)', fontsize=16, y=1.02)
plt.show()

## 3. The Saturation Reveal (The "Aha" Moment)
Our features were built to react to changes in promotional intensity. But what happens if the feature loses variance? 
Let's look at the Promo Intensity specifically focused on our 12-week test holdout.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df_weekly['Week'], df_weekly['Promo'], color='#1f77b4', linewidth=2)

holdout_start = df_weekly['Week'].iloc[-12]
holdout_end = df_weekly['Week'].iloc[-1]

ax.axvspan(holdout_start, holdout_end, color='red', alpha=0.2, label='12-Week Holdout Period')
saturation_level = df_weekly['Promo'].iloc[-12:].mean()

ax.set_title(f'Feature Degeneracy: Promo Saturation ({saturation_level:.1%} Intensity)', fontweight='bold')
ax.set_xlabel('Week Number')
ax.set_ylabel('% of Stores on Promotion')
ax.legend()
plt.show()

### The Implication
During the holdout, practically every store was constantly on promotion. When external regressors flatline (var = 0), a time series degrades into a pure autoregressive process. **A Naive ($y_t = y_{t-1}$) model will therefore dominate ML models on chain-level aggregates.** To build an ML model that beats the baseline, we must predict at the granular store-level where promotional variance still exists week-over-week.